Loads all of the libraries and the source that we will be using.


In [2]:
import polars as pl
import matplotlib.pyplot as plt
from tslearn.clustering import TimeSeriesKMeans
from sklearn.metrics import silhouette_score
import os
source = "cleaned_water_usage_2023-08.parquet"

"""
Spring: March, April, May
Summer: June, July, Aug
Fall: Sept, Oct, Nov
Winter: Dec, Jan, Feb 

"""

'\nSpring: March, April, May\nSummer: June, July, Aug\nFall: Sept, Oct, Nov\nWinter: Dec, Jan, Feb \n\n'

Need to add paralization either though CPU core utilization or though GPU CUDA 


In [3]:
if not os.path.exists(source):
    print(f"File not found: {source}")
else:
    df_month = pl.read_parquet(source)
    df_sorted = df_month.sort("meter_uuid", "timestamp_UTC")

    # Aggregate to daily sums for each meter using the sorted DataFrame
    df_daily = df_sorted.group_by_dynamic(
        "timestamp_UTC", 
        every="1d",         # Group by each day
        by="meter_uuid"
    ).agg(
        pl.sum("water_usage_m3")
    ).with_columns(
        # Create a simple day column (e.g., 1 to 31) for pivoting
        day=pl.col("timestamp_UTC").dt.day()
    )

    # --- Step 2: Reshape Data from Long to Wide (Pivot) ---
    df_wide = df_daily.pivot(
        index="meter_uuid", 
        columns="day",
        values="water_usage_m3"
    )

    # --- Step 3: Handle Missing Values and Convert to NumPy ---
    df_filled = df_wide.fill_null(0)

    meter_ids = df_filled["meter_uuid"]  
    time_series_data = df_filled.drop("meter_uuid").to_numpy() 

    print(f"Data reshaped. Found {time_series_data.shape[0]} meters with {time_series_data.shape[1]} days of data.")

    # --- Step 4: Determine the Optimal Number of Clusters (Elbow Method) ---
    print("\nCalculating inertia for different numbers of clusters (Elbow Method)...")
    distortions = []
    K_range = range(2, 10)

    for k in K_range:
        km = TimeSeriesKMeans(n_clusters=k, metric="dtw", max_iter=5, random_state=42, n_jobs=-1)
        km.fit(time_series_data)
        distortions.append(km.inertia_)

    plt.figure(figsize=(10, 6))
    plt.plot(K_range, distortions, 'bo-')
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia (Sum of Squared Distances)')
    plt.title('Elbow Method for Optimal k')
    plt.show()

    OPTIMAL_K = 4 # Look at the plot and change this value
    print(f"Based on the plot, we will proceed with k={OPTIMAL_K} clusters.")

    # --- Step 5: Run the K-Means with DTW Clustering ---
    print("\nRunning final clustering with DTW...")
    final_km = TimeSeriesKMeans(n_clusters=OPTIMAL_K, metric="dtw", max_iter=10, random_state=42, n_jobs=-1)
    cluster_labels = final_km.fit_predict(time_series_data)

    # --- Step 6: Analyze and Visualize the Results ---
    results = pl.DataFrame({
        "meter_uuid": meter_ids, 
        "cluster": cluster_labels
    })
    print("\nClustering Results:")
    print(results["cluster"].value_counts(sort=True))

    plt.figure(figsize=(15, 10))
    for i in range(OPTIMAL_K):
        plt.subplot(OPTIMAL_K, 1, i + 1)
        cluster_series = time_series_data[cluster_labels == i]
        for series in cluster_series:
            plt.plot(series.ravel(), "k-", alpha=.2)
        plt.plot(final_km.cluster_centers_[i].ravel(), "r-")
        plt.title(f"Cluster {i} ({len(cluster_series)} meters)")
        plt.ylabel("Daily Water Usage (m3)")

    plt.tight_layout()
    plt.xlabel(f"Day of the Month")
    plt.show()

File not found: cleaned_water_usage_2023-08.parquet


Saves the cluster assignments to a csv file.

In [ ]:
meters_by_cluster = results.group_by("cluster").agg(
    pl.col("meter_uuid").alias("meters_in_this_cluster")
).sort("cluster")

print(meters_by_cluster)
output_csv_path = "meter_cluster_assignments_2021-12.csv"

print(f"\n--- Saving full results to {output_csv_path} ---")
results.write_csv(output_csv_path)

print("File saved successfully.")

Plots the Cluster's Centriods

In [ ]:
plt.figure(figsize=(12, 7))
for i in range(OPTIMAL_K):
    # Each centroid is a 1D array, so .ravel() flattens it
    plt.plot(final_km.cluster_centers_[i].ravel(), label=f'Cluster {i} Centroid')

plt.title('Combined Centroids of All Water Usage Clusters')
plt.xlabel('Day of the Month (December 2021)')
plt.ylabel('Average Daily Water Usage (m3)')
plt.legend()
plt.grid(True)
plt.show()
